#Cruce de tablas para la obtención de correos de personal
Este proyecto tiene como finalidad cruzar tres fuentes de datos para la obtención de correos de personal:


1. La lista de servidores vigentes enviadas por OAPC.
2. La lista de correos electrónicos del Google Workspace enviada por la coordinación de redes.
3. La lista de cuentas de red del dictorio activo enviada por la coordinación de redes.

A continuación se realiza el proyecto siguiendo las etapas de extracción, transformación, cruce de las fuentes de información y exportación de la lista de correos del personal vigente en la entidad.

##Extracción

**Carga de dependencias:** Importar pandas, numpy, unicodedata, re  (regular expressions) y los módulos process y fuzz de rapidfuzz.


In [1]:
#Se instala e importa chardet para detectar el encoding del archivo CSV.
!pip install chardet
import chardet

#Se importa pandas para realizar el ETL usando dataframes.
import pandas as pd

#Se importa numpy como dependencia usada por algunas funciones de pandas.
import numpy as np

#Se importa unicodedata para la eliminación de tildes.
import unicodedata

#Se importa re para el uso de expresiones regulares en la normalización de nombres y apellidos.
import re

#Se instala rapidfuzz e importan fuzz y process para el cálculo y búsqueda de coincidencias difusas.
!pip install rapidfuzz
from rapidfuzz import process, fuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.6 MB/s eta 0:00:00


**Ingesta con tipado estricto:** Cargar la Tabla 1 (Excel), Tabla 2 (Google) y Tabla 3 (AD) utilizando pd.read_excel o pd.read_csv. Primero, se detectó el tipo de encoding del archivo csv y luego se procedió con la carga de los datos en dataframes.

Nota: Se utilizó el parámetro `dtype={'DNI': str}` para evitar que se pierdan los ceros a la izquierda.

In [2]:
#Se crea una lista con las ubicaciones del archivo csv del DA.
ruta_1 = '/content/tabla1_personal.csv'
ruta_2 = '/content/tabla2_workspace.csv'
ruta_3 = '/content/tabla3_directorio.csv'

rutas = [ruta_1, ruta_2, ruta_3]

#Se crea un bucle for para saber el encoding de cada archivo.
for ruta_archivo in rutas:
  #Se abre el archivo como binario.
  with open(ruta_archivo, 'rb') as f:
      data_bruto = f.read(10000) #Se leen los primero 10000 bytes.
      resultado = chardet.detect(data_bruto)['encoding'] #Se detecta el encoding.

  #Se imprime el encoding del archivo CSV.
  print(f"Encoding detectado: {resultado}")

Encoding detectado: UTF-8-SIG
Encoding detectado: UTF-8-SIG
Encoding detectado: UTF-8-SIG


In [3]:
#Se importa la tabla 1 en el dataframe 1.
df_1 = pd.read_csv("/content/tabla1_personal.csv", encoding='UTF-8-SIG', dtype={'DNI': str}) #Nos aseguramos que el DNI se trate como cadena de texto.

#Se importa la tabla 2 en el dataframe 2.
df_2 = pd.read_csv("/content/tabla2_workspace.csv", encoding='UTF-8-SIG')

#Se importa la tabla 3 en el dataframe 3.
df_3 = pd.read_csv("/content/tabla3_directorio.csv", encoding='UTF-8-SIG', dtype={'Description': str}) #Nos aseguramos que Description (DNI) se trate como cadena de texto.

#Se imprime la cantidad de filas por cada df.
for i in [df_1, df_2, df_3]:
  print(f"Cantidad de filas: {len(i)}")

Cantidad de filas: 1050
Cantidad de filas: 1430
Cantidad de filas: 1320


**Auditoría de tipos:** Verificar que los campos tengan el tipo de datos correspondiente y realizar los cambios que se requiera usando pandas.

Nota: `Last Sign In [READ ONLY]` (de la tabla 2) y `CORREO ELECTRONICO INSTITUCIONAL` (de la tabla 1) son los únicos campo cuyo tipo es incorrecto. Sin embargo, como se eliminarán, no se corregirán sus tipo de dato.

In [5]:
#Se imprime el tipo de datos que contienen los campos de cada df.
for i, df in enumerate([df_1, df_2, df_3]):
    print("-" * 30)
    print(f"Daframe N° {i+1}")
    print("-" * 30)
    print(df.dtypes)
    print("\n") # Espacio extra entre resultados

------------------------------
Daframe N° 1
------------------------------
N°                                          int64
DNI                                        object
APELLIDOS                                  object
NOMBRES                                    object
CARGO                                      object
FECHA DE INICIO                            object
REGIMEN LABORAL                            object
ÓRGANO                                     object
UNIDAD ORGANICA                            object
ESTADO                                     object
SERVIDORES CIVILES QUE TIENEN LICENCIA     object
CORREO ELECTRONICO INSTITUCIONAL          float64
APELLIDOS Y NOMBRES                        object
dtype: object


------------------------------
Daframe N° 2
------------------------------
First Name [Required]       object
Last Name [Required]        object
Email Address [Required]    object
Status [READ ONLY]          object
Last Sign In [READ ONLY]    object
Storage U

##Limpieza

**Depuración de columnas:** Eliminar columnas irrelevantes (cargos, fechas, almacenamiento) para optimizar el uso de memoria y facilitar los cruces.

In [6]:
#Se eliminan las columnas irrelevantes del df1
cols_eliminar_df1 = ['N°', 'CARGO', 'FECHA DE INICIO', 'REGIMEN LABORAL', 'ÓRGANO', 'UNIDAD ORGANICA', 'ESTADO', 'SERVIDORES CIVILES QUE TIENEN LICENCIA', 'CORREO ELECTRONICO INSTITUCIONAL']
df_1.drop(columns=cols_eliminar_df1, inplace=True)

#Se eliminan las columnas irrelevantes del df2
cols_eliminar_df2 = ['Status [READ ONLY]', 'Last Sign In [READ ONLY]', 'Storage Used [READ ONLY]']
df_2.drop(columns=cols_eliminar_df2, inplace=True)

#Se eliminan las columnas irrelevantes del df3
cols_eliminar_df3 = ['Name', 'Disabled', 'Username (pre 2000)']
df_3.drop(columns=cols_eliminar_df3, inplace=True)

In [7]:
#Se imprime nuevamente los campos de cada df para comprobar los cambios.
for i, df in enumerate([df_1, df_2, df_3]):
    print("-" * 30)
    print(f"Daframe N° {i+1}")
    print("-" * 30)
    print(df.dtypes)
    print("\n") # Espacio extra entre resultados

------------------------------
Daframe N° 1
------------------------------
DNI                    object
APELLIDOS              object
NOMBRES                object
APELLIDOS Y NOMBRES    object
dtype: object


------------------------------
Daframe N° 2
------------------------------
First Name [Required]       object
Last Name [Required]        object
Email Address [Required]    object
dtype: object


------------------------------
Daframe N° 3
------------------------------
Description      object
Email Address    object
First Name       object
Last Name        object
dtype: object




**Contar registros nulos (tabla 1):** Contar los registros (filas) que tienen valores nulos (NaN o None) en la columna específica "APELLIDOS Y NOMBRES". En caso hayan nulos, comprobar si tiene también valor nulo en la columna "DNI" y eliminar si es así. Caso contrario, mantener el registro.

In [8]:
# Usamos isna() para detectar nulos y sum() para contarlos
nulos = df_1['APELLIDOS Y NOMBRES'].isna().sum()
print(f"Cantidad de nulos en la columna APELLIDOS Y NOMBRES: {nulos}")

Cantidad de nulos en la columna APELLIDOS Y NOMBRES: 0


In [9]:
# Usamos isna() para detectar nulos y sum() para contarlos
nulos = df_1['DNI'].isna().sum()
print(f"Cantidad de nulos en la columna DNI: {nulos}")

Cantidad de nulos en la columna DNI: 0


**Estandarización de DNI:** Aplicar `df['DNI'].str.zfill(8)` en la Tabla 1 y Tabla 3 para asegurar que todos tengan 8 dígitos.

In [10]:
#Se completa con ceros a la izquierda los valores de DNI (df1) con menos de 8 digitos.
df_1['DNI']=df_1['DNI'].str.zfill(8)

#Se completa con ceros a la izquierda los valores de Description (df3) con menos de 8 digitos.
df_3['Description']=df_3['Description'].str.zfill(8)

**Normalización de texto universal:** Crear una función para limpiar las columnas de nombres y apellidos en las tres tablas que realice:  

- Conversión a mayúsculas (str.upper()).

- Eliminación de tildes y diacríticos (usando unicodedata).

- Eliminación de espacios dobles y espacios en los extremos (usando regex y str.strip()).

In [11]:
#Función para limpieza profunda de nombres y apellidos.
def normalizar_identidad(texto):
    #Se comprueba si hay valores perdidos (Nan/None) o no hay texto, y retorna lo mismo.
    if pd.isna(texto) or texto == "":
        return texto

    #Si no es vacío se realizan las siguientes transformaciones.
    #1. Convertir a string y a Mayúsculas
    texto = str(texto).upper()

    #2. Eliminar tildes
    # Normaliza a NFKD (separa caracteres de sus tildes) y codifica a ASCII ignorando errores (producidos por las tildes).
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')

    #3. Regex para eliminar espacios innecesarios dejando uno solo,y strip para los extremos.
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [12]:
#Dataframe 1 (RH): Columna 'APELLIDOS Y NOMBRES'
df_1['APELLIDOS Y NOMBRES'] = df_1['APELLIDOS Y NOMBRES'].apply(normalizar_identidad)

#Dataframe 2 (Workspace): Columnas 'First Name' y 'Last Name'
df_2['First Name [Required]'] = df_2['First Name [Required]'].apply(normalizar_identidad)
df_2['Last Name [Required]'] = df_2['Last Name [Required]'].apply(normalizar_identidad)

#Dataframe 3 (Directorio Activo): Columna 'Nombre y apellidos'
df_3['First Name'] = df_3['First Name'].apply(normalizar_identidad)
df_3['Last Name'] = df_3['Last Name'].apply(normalizar_identidad)

**Descomposición de Identidad (Tabla 1):** Utilizar str.split(n=2, expand=True) en la columna "APELLIDOS Y NOMBRES" para crear las columnas "Apellido Paterno", "Apellido Materno" y "Nombres".

La función `separar_apellidos_nombres_robusto` identifica las palabras que unen los apellidos, y realiza un conteo inteligente hasta identificar todos los apellidos posibles considerando la última palabra como el nombre. A groso modo, se manejan dos casos:
1. Las 2 primeras palabras del texto son los apellidos.
2. Los apellidos, los dos o uno de ellos, contiene partículas (palabras de unión).

*Nota:* Como la separación no es perfecta, se dejará la columna original "APELLIDOS Y NOMBRES" de la tabla 1 en caso se necesite hacer una búsqueda manual.

In [13]:
def separar_apellidos_nombres_robusto(texto):

    #Definimos las partículas de unión comunes en apellidos hispanos.
    particulas = {'DE', 'LA', 'LAS', 'EL', 'LOS', 'DEL'}

    palabras = texto.split()
    n = len(palabras)

    #Si solo hay una palabra, se asume como apellido.
    if n == 1:
        return pd.Series([palabras[0], ""])

    # Lógica de conteo inteligente para identificar el límite de los apellidos
    # Por defecto, intentamos buscar 2 apellidos.
    indice_corte = 0 #Se refiere a la cantidad de palabras examinadas.
    apellidos_contados = 0

    i = 0
    #Se buscan apellidos mientras no se hayan acabado las palabras y el número de apellidos sea menor a 2.
    while i < n and apellidos_contados < 2:
        # Si la palabra actual es una partícula, no cuenta como "apellido completo"
        # sino como unión, por lo que saltamos a la siguiente sin sumar al contador
        if palabras[i] in particulas:
            i += 1
            # Manejo especial para dos párticulas seguidas (ej. "DE LA").
            if i < n and palabras[i] in particulas:
                i += 1
        else:
            # Es una palabra de apellido real (ej. "VASQUEZ", "CRUZ", etc).
            i += 1
            apellidos_contados += 1

        indice_corte = i

        # Si después de encontrar el primer o segundo apellido solo queda una palabra,
        # esa última se considera el nombre.
        if n - indice_corte == 1:
            break

    #Se unen los textos en apellidos y nombres usando el índice de corte, y separando cada palabra con un espacio.
    apellidos = " ".join(palabras[:indice_corte])
    nombres = " ".join(palabras[indice_corte:])

    return pd.Series([apellidos, nombres])

In [14]:
#Separación de "APELLIDOS Y NOMBRES" en dos columnas "APELLIDOS" y "NOMBRES".
df_1[['APELLIDOS', 'NOMBRES']] = df_1['APELLIDOS Y NOMBRES'].apply(separar_apellidos_nombres_robusto)

In [15]:
len(df_1)

1050

**Identificación de duplicados (Tabla 1):** Utilizar una mascara booleana para indentificar los registros con valores en DNI duplicados. Existen duplicados porque una persona puede tener más de un cargo.

In [17]:
#Se busca DNI duplicados en la Tabla 1 (OGRH), el parámetro Keep sirve para marcar todos los duplicados como True.
duplicados_t1 = df_1[df_1.duplicated(subset=['DNI'], keep=False)]

#Imprimimos la cantidad de duplicados.
print(f"Registros duplicados en Tabla 1: {len(duplicados_t1)}")

#Se muestran los valores duplicados, ordenandolo según DNI.
duplicados_t1.head()

Registros duplicados en Tabla 1: 100


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES
9,21429111,MANCEBO BERNAL,RITA JOAN,MANCEBO BERNAL RITA JOAN
29,85899314,FONT MESA,SABAS HILDA,FONT MESA SABAS HILDA
42,84214383,SANZ MACHADO,TEO JOSEFINA,SANZ MACHADO TEO JOSEFINA
47,72070938,ARINO BILBAO,SIMON GEMA,ARINO BILBAO SIMON GEMA
56,15014632,LAGUNA VEGA,AMAYA JOSE,LAGUNA VEGA AMAYA JOSE


**Eliminación de duplicados (Tabla 1):** Necesitamos quitar los duplicados para establecer el DNI como índice (que no se puede repetir) a partir de cual mapeamos los correos electrónicos desde los df cruzados hacia un df consolidado.

In [18]:
#Se eliminan los duplicados en Tabla 1 manteniendo solo la primera aparición.
df_1_unicos = df_1.drop_duplicates(subset=['DNI'], keep='first').copy()
len(df_1_unicos)

1000

**Identificación de duplicados (Tabla 3):** Utilizar una mascara booleana para indentificar los registros con valores en Description (DNI) duplicados.

In [19]:
#Se busca DNI duplicados en la Tabla 3 (Directorio Activo), el parámetro Keep sirve para marcar todos los duplicados como True.
duplicados_t3 = df_3[df_3.duplicated(subset=['Description'], keep=False)]

#Imprimimos la cantidad de duplicados.
print(f"Registros duplicados en Tabla 3: {len(duplicados_t3)}")

#Se muestran los valores duplicados, ordenandolo según DNI.
duplicados_t3.head()

Registros duplicados en Tabla 3: 745


,Description,Email Address,First Name,Last Name
1,18533118,lmata@dummie.com,LOPE MARIA DEL CARMEN,MATA BOLANOS
9,00MOISÉS,agómez@dummie.com,AINARA HAYDEE,GOMEZ CASES
17,00MOISÉS,dllanos@dummie.com,DOMINGO CONSUELA,LLANOS MAESTRE
20,00MOISÉS,xmárquez@dummie.com,XAVIER MODESTO,MARQUEZ PI
22,00021429,rmancebo@dummie.com,RITA JOAN,MANCEBO BERNAL


**Eliminación de duplicados (Tabla3):** Para mantener la integridad de la Tabla 1 (Personal Activo) y no alterar la cantidad de registro, se eliminan los duplicados de la Tabla 3 antes de realizar el cruce prefiriendo el primer registro encontrado.

In [20]:
#Se eliminan los duplicados en Tabla 3 manteniendo solo la primera aparición.
df_3_unicos = df_3.drop_duplicates(subset=['Description'], keep='first').copy()

**Identificación de duplicados (Tabla 2):** Utilizar una mascara booleana para indentificar los registros con valores en "First Name [Required]" y "Last Name [Required]" duplicados.

In [21]:
#Se busca duplicados en la Tabla 2 (Google Workspace) en las columnas de nombre y apellido.
duplicados_t2 = df_2[df_2.duplicated(subset=['First Name [Required]', 'Last Name [Required]'], keep=False)]

#Imprimimos la cantidad de duplicados.
print(f"Registros duplicados en Tabla 2: {len(duplicados_t2)}")

#Se muestran los valores duplicados, ordenandolos por nombre y apellido.
duplicados_t2.head()

Registros duplicados en Tabla 2: 298


,First Name [Required],Last Name [Required],Email Address [Required]
1,NEREA SALOME,BARROS CAL,nbarros@dummie.com
17,JULIE BUENAVENTURA,PARDO FERNANDEZ,jpardo@dummie.com
20,AMOR ROBERTO,PRAT VALDERRAMA,aprat@dummie.com
28,MARGARITA ISABELA,LEON ESPANA,mleon@dummie.com
36,CANDELA ELEUTERIO,MUR SANCHEZ,cmur@dummie.com


**Eliminación de duplicados (Tabla 2):** Para mantener la integridad de la Tabla 1 (Personal Activo) y no alterar la cantidad de registro, se eliminan los duplicados de la Tabla 2 antes de realizar el cruce prefiriendo el primer registro encontrado.

In [22]:
#Se eliminan los duplicados en Tabla 2 manteniendo solo la primera aparición.
df_2_unicos = df_2.drop_duplicates(subset=['First Name [Required]', 'Last Name [Required]'], keep='first').copy()

##Cruce

**Consulta 1 - Cruce por DNI:** Ejecutar un pd.merge tipo Left Join entre la Tabla 1 (unicos) y la Tabla 3 (unicos) usando el DNI como llave.

In [23]:
#Se realiza un left join entre df_1 y df_3 usando DNI y Description como llave.
df_cruzado_dni = pd.merge(df_1_unicos, df_3_unicos, left_on='DNI', right_on='Description', how='left')

#Se muestran las primeras filas del dataframe resultante y su tamaño para verificar el cruce.
print("Columnas del Dataframe cruzado:")
print(df_cruzado_dni.dtypes)

Columnas del Dataframe cruzado:
DNI                    object
APELLIDOS              object
NOMBRES                object
APELLIDOS Y NOMBRES    object
Description            object
Email Address          object
First Name             object
Last Name              object
dtype: object


In [24]:

#Se eliminan las columnas con información redundante
cols_redundates = ["First Name", "Last Name", "Description"]
df_cruzado_dni.drop(columns=cols_redundates, inplace=True)
#Se muestran las primeras filas del dataframe resultante y su tamaño para validar el cruce.
print("Primeras filas del dataframe cruzado por DNI:")
display(df_cruzado_dni[:5])
print(f"\nNúmero de filas en el dataframe cruzado por DNI: {len(df_cruzado_dni)}")

Primeras filas del dataframe cruzado por DNI:


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address
0,85822413,CANTON AMAYA,MANUELA LEONARDO,CANTON AMAYA MANUELA LEONARDO,mcantón@dummie.com
1,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,NaN
2,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN
3,13756670,RIVAS GIMENEZ,VIDAL DOLORES,RIVAS GIMENEZ VIDAL DOLORES,vrivas@dummie.com
4,56629389,CUERVO ROCAMORA,NOE LEOPOLDO,CUERVO ROCAMORA NOE LEOPOLDO,NaN



Número de filas en el dataframe cruzado por DNI: 1000


**Segmentación de resultados:** Crear un DataFrame de "No Encontrados" filtrando aquellos registros que resultaron con valor Nan en el campo de correo electrónico tras la Consulta 1.

In [25]:
#Convertimos strings vacíos o con puros espacios a NaN.
df_cruzado_dni['Email Address'] = df_cruzado_dni['Email Address'].replace(r'^\s*$', np.nan, regex=True)

#Se filtran los registros con valores nulos de la columna Email Address.
nulos_dni = df_cruzado_dni[df_cruzado_dni['Email Address'].isna()].copy()

#Imprimimos la cantidad de nulos en el primer cruce.
print(f"Registros sin correo tras el primer cruce {len(nulos_dni)}")

Registros sin correo tras el primer cruce 524


**Consulta 2 - Cruce Nominal:** Realizar un pd.merge entre el subconjunto de "No Encontrados" de la consulta 1 y la Tabla 2 (unicos) utilizando como llaves de cruce los campos de Apellidos y Nombres normalizados.

In [26]:
#Se realiza el merge nominal entre nulos_dni y df_2_unicos.
#Usamos left join para mantener todos los registros de 'nulos_dni'.
df_cruzado_nominal = pd.merge(
    nulos_dni,
    df_2_unicos,
    left_on=['APELLIDOS', 'NOMBRES'],
    right_on=['Last Name [Required]', 'First Name [Required]'],
    how='left'
)

#Al principio, la columna 'Email Address [Required]' de df_2 contendrá los correos encontrados.
#La columna 'Email Address' de nulos_dni sigue siendo NaN.
#Rellenamos la columna 'Email Address' con los valores de 'Email Address [Required]' donde haya coincidencia.
#Si no hay coincidencia, 'Email Address [Required]' también será NaN.
df_cruzado_nominal['Email Address'] = df_cruzado_nominal['Email Address'].fillna(
    df_cruzado_nominal['Email Address [Required]']
)

#Se muestran las primeras filas del dataframe resultante y su tamaño para verificar el cruce.
print("Columnas del Dataframe cruzado:")
print(df_cruzado_nominal.dtypes)

Columnas del Dataframe cruzado:
DNI                         object
APELLIDOS                   object
NOMBRES                     object
APELLIDOS Y NOMBRES         object
Email Address               object
First Name [Required]       object
Last Name [Required]        object
Email Address [Required]    object
dtype: object


In [27]:
#Se eliminan las columnas redundantes de df_2 que se usaron para el cruce.
cols_redundates = ['First Name [Required]', 'Last Name [Required]', 'Email Address [Required]']
df_cruzado_nominal.drop(columns=cols_redundates, inplace=True)

# Imprimir las primeras filas y el tamaño del DataFrame resultante para verificar
print("Primeras filas del dataframe cruzado nominal:")
display(df_cruzado_nominal[:5])
print(f"\nNúmero de filas en el dataframe cruzado nominal: {len(df_cruzado_nominal)}")

Primeras filas del dataframe cruzado nominal:


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address
0,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,NaN
1,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN
2,56629389,CUERVO ROCAMORA,NOE LEOPOLDO,CUERVO ROCAMORA NOE LEOPOLDO,ncuervo@dummie.com
3,12575563,NAVARRO CALATAYUD,CIRINO RICARDO,NAVARRO CALATAYUD CIRINO RICARDO,cnavarro@dummie.com
4,67827639,LAMAS AROCA,LUNA ANGEL,LAMAS AROCA LUNA ANGEL,NaN



Número de filas en el dataframe cruzado nominal: 524


**Segmentación de resultados:** Crear un DataFrame de "No Encontrados" filtrando aquellos registros que resultaron con valor Nan en el campo de correo electrónico tras la Consulta 2. No hay valores vacíos, porque la tabla 2 procede de Workspace donde el campo correo electrónico siempre está lleno.

In [28]:
#Cálculo de registros vacíos no detectados por isnan.
vacio = (df_cruzado_nominal['Email Address'].str.strip()=="").sum()
print(f'Registros sin correo tras el segundo cruce {vacio} no detectados por isnan.')

Registros sin correo tras el segundo cruce 0 no detectados por isnan.


In [29]:
#Se filtran los registros con valores nulos de la columna Email Address.
#Al añadir .copy(), creas un DataFrame nuevo e independiente en memoria.
#Es importante crearlo como df independiente para añadir los resultados de la busqueda difusa en la tabla cruzada 3.
nulos_nominal = df_cruzado_nominal[df_cruzado_nominal['Email Address'].isna()].copy()

#Imprimimos la cantidad de nulos en el primer cruce.
print(f"Registros sin correo tras el segundo cruce {len(nulos_nominal)}")

Registros sin correo tras el segundo cruce 317


In [30]:
nulos_nominal.dtypes

,0
DNI,object
APELLIDOS,object
NOMBRES,object
APELLIDOS Y NOMBRES,object
Email Address,object


**Consulta 3 - Fuzzy Matching con DA (Coincidencia Difusa):** Para los registros que aún persisten sin correo:
* Generar una función para la búsqueda difusa.

* Generar una lista de nombres completos normalizados de la Tabla 3.

* Iterar sobre los faltantes de la consulta 2 usando process.extractOne de RapidFuzz.

* Aplicar un score_cutoff=90 para validar automáticamente solo las coincidencias de alta confianza.

In [31]:
#Función para realizar la búsqueda difusa, que devuelve la "Mejor coincidencia" y "Puntaje".
def aplicar_fuzzy_matching(nombre, lista_referencia):
    #Se usa extractOne para encontrar la mejor coincidencia.
    #"score_cutoff=90" asegura que solo aceptamos coincidencias casi exactas.
    resultado = process.extractOne(
        nombre,
        lista_referencia,
        scorer=fuzz.token_sort_ratio, #Ignora el orden de las palabras.
        score_cutoff=90
    )

    # resultado devuelve (mejor_coincidencia, puntaje) o None
    if resultado:
      mejor_coincidencia = resultado[0]
      puntaje = resultado[1]
      return pd.Series([mejor_coincidencia, puntaje]) #Se pasa el resultado como una lista.

    return pd.Series([None, 0])

In [32]:
#Combinamos nombres y apellidos del df 3 (Directorio Activo) para crear el campo de comparación.
#Se usa .loc para asignar la columna a todas las filas (:)
df_3_unicos.loc[:, 'Full name'] = df_3_unicos['Last Name'] + " " + df_3_unicos['First Name']

#Se pasan los valores a una lista para utilizarla en la búsqueda difusa con RapidFuzz.
lista_referencia_da = df_3_unicos['Full name'].tolist()

In [33]:
#Se aplica la función, creando dos columnas: la coincidencia hallada y su puntaje
nulos_nominal[['Mejor_coincidencia', 'Puntaje']] = nulos_nominal['APELLIDOS Y NOMBRES'].apply(
    lambda x: aplicar_fuzzy_matching(x, lista_referencia_da))

In [34]:
nulos_nominal.head()

,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address,Mejor_coincidencia,Puntaje
0,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,NaN,CALATAYUD BONET DOMINGA TELMO,100.0
1,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN,NaN,0.0
4,67827639,LAMAS AROCA,LUNA ANGEL,LAMAS AROCA LUNA ANGEL,NaN,NaN,0.0
5,26687538,PINEDO BERMUDEZ,CARMINA LAZARO,PINEDO BERMUDEZ CARMINA LAZARO,NaN,PINEDO BERMUDEZ CARMINA LAZARO,100.0
6,21429111,MANCEBO BERNAL,RITA JOAN,MANCEBO BERNAL RITA JOAN,NaN,MANCEBO BERNAL RITA JOAN,100.0


In [35]:
#Se recupera el "correo electrónico" de la Tabla 3 basado en la "Mejor coincidencia" de la Tabla 1.
#Se hace un merge para traer el correo de la fila que RapidFuzz identificó como mejor coincidencia.
df_cruzado_difuso_da = pd.merge(
    nulos_nominal,
    df_3_unicos[['Full name','Email Address']],
    left_on='Mejor_coincidencia',
    right_on='Full name',
    how='left'
)

#Se muestran el tipado del dataframe resultante y su tamaño para verificar el cruce.
print("Columnas del Dataframe cruzado:")
print(df_cruzado_difuso_da.dtypes)

Columnas del Dataframe cruzado:
DNI                     object
APELLIDOS               object
NOMBRES                 object
APELLIDOS Y NOMBRES     object
Email Address_x         object
Mejor_coincidencia      object
Puntaje                float64
Full name               object
Email Address_y         object
dtype: object


In [36]:
#Rellenamos la columna 'Email Address_x' con los valores de 'Email Address_y' donde haya coincidencia.
#Si no hay coincidencia, 'Email Address_x' también será NaN.
df_cruzado_difuso_da['Email Address_x'] = df_cruzado_difuso_da['Email Address_x'].fillna(
    df_cruzado_difuso_da['Email Address_y']
)

#Renombramos 'Email Address_x' como 'Email Address'.
df_cruzado_difuso_da.rename(columns={'Email Address_x': 'Email Address'}, inplace=True)

#Se eliminan las columnas redundantes que quedaron tras el cruce.
cols_redundates = ['Mejor_coincidencia', 'Puntaje', 'Full name', 'Email Address_y']
df_cruzado_difuso_da.drop(columns=cols_redundates, inplace=True)

# Imprimir las primeras filas y el tamaño del DataFrame resultante para verificar
print("Primeras filas del dataframe cruzado difuso DA:")
display(df_cruzado_difuso_da[:5])
print(f"\nNúmero de filas en el dataframe cruzado difuso: {len(df_cruzado_difuso_da)}")

Primeras filas del dataframe cruzado difuso DA:


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address
0,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,dcalatayud@dummie.com
1,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN
2,67827639,LAMAS AROCA,LUNA ANGEL,LAMAS AROCA LUNA ANGEL,NaN
3,26687538,PINEDO BERMUDEZ,CARMINA LAZARO,PINEDO BERMUDEZ CARMINA LAZARO,cpinedo@dummie.com
4,21429111,MANCEBO BERNAL,RITA JOAN,MANCEBO BERNAL RITA JOAN,rmancebo@dummie.com



Número de filas en el dataframe cruzado difuso: 317


**Segmentación de resultados:** Crear un DataFrame de "No Encontrados" filtrando aquellos registros que resultaron con valor Nan o vacío en el campo de correo electrónico tras la Consulta 3.

In [37]:
#Convertimos strings vacíos o con puros espacios a NaN.
df_cruzado_difuso_da['Email Address'] = df_cruzado_difuso_da['Email Address'].replace(r'^\s*$', np.nan, regex=True)

#Se filtran los registros con valores nulos de la columna Email Address.
nulos_difuso_da = df_cruzado_difuso_da[df_cruzado_difuso_da['Email Address'].isna()].copy()

#Imprimimos la cantidad de nulos en el tercer cruce.
print(f"Registros sin correo tras el tercer cruce {len(nulos_difuso_da)}")

Registros sin correo tras el tercer cruce 213


**Consulta 4 - Fuzzy Matching con Workspace (Coincidencia Difusa):** Para los registros que aún persisten sin correo:
* Generar una lista de apellidos normalizados de la Tabla 2. El nombre completo podría no generar una coincidencia superior al umbral mínimo admitido.

* Iterar sobre los faltantes de la consulta 3 usando process.extractOne de RapidFuzz.

* Aplicar un score_cutoff=90 para validar automáticamente solo las coincidencias de alta confianza.

In [38]:
#Se pasan los valores a una lista para utilizarla en la búsqueda difusa con RapidFuzz.
lista_referencia_workspace = df_2_unicos['Last Name [Required]'].tolist()

In [39]:
#Se aplica la función, creando dos columnas: la coincidencia hallada y su puntaje
nulos_difuso_da[['Mejor_coincidencia', 'Puntaje']] = nulos_difuso_da['APELLIDOS'].apply(
    lambda x: aplicar_fuzzy_matching(x, lista_referencia_workspace))

In [40]:
nulos_difuso_da.head()

,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address,Mejor_coincidencia,Puntaje
1,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN,NaN,0.000000
2,67827639,LAMAS AROCA,LUNA ANGEL,LAMAS AROCA LUNA ANGEL,NaN,LAMAS AROC,95.238095
6,25808538,CUESTA ROCA,DEBORA BERNARDO,CUESTA ROCA DEBORA BERNARDO,NaN,NaN,0.000000
7,88753261,ANGEL IGLESIAS,CAROLINA ROSALINDA,ANGEL IGLESIAS CAROLINA ROSALINDA,NaN,NaN,0.000000
8,10709498,CARBO LASA,MARIA JOSE FELIPA,CARBO LASA MARIA JOSE FELIPA,NaN,NaN,0.000000


In [41]:
#Se recupera el "correo electrónico" de la Tabla 2 basado en la "Mejor coincidencia" de la Tabla 1.
#Se hace un merge para traer el correo de la fila que RapidFuzz identificó como mejor coincidencia.
df_cruzado_difuso_workspace = pd.merge(
    nulos_difuso_da,
    df_2_unicos[['Last Name [Required]','Email Address [Required]']],
    left_on='Mejor_coincidencia',
    right_on='Last Name [Required]',
    how='left'
)

#Se muestran las primeras filas del dataframe resultante y su tamaño para verificar el cruce.
print("Columnas del Dataframe cruzado:")
print(df_cruzado_difuso_workspace.dtypes)

Columnas del Dataframe cruzado:
DNI                          object
APELLIDOS                    object
NOMBRES                      object
APELLIDOS Y NOMBRES          object
Email Address                object
Mejor_coincidencia           object
Puntaje                     float64
Last Name [Required]         object
Email Address [Required]     object
dtype: object


In [42]:
#Rellenamos la columna 'Email Address' con los valores de 'Email Address [Required]' donde haya coincidencia.
#Si no hay coincidencia, 'Email Address' también será NaN o vacío.
df_cruzado_difuso_workspace['Email Address'] = df_cruzado_difuso_workspace['Email Address'].fillna(
    df_cruzado_difuso_workspace['Email Address [Required]']
)

#Se eliminan las columnas redundantes que quedaron tras el cruce.
cols_redundates = ['Last Name [Required]', 'Email Address [Required]']
df_cruzado_difuso_workspace.drop(columns=cols_redundates, inplace=True)

# Imprimir las primeras filas y el tamaño del DataFrame resultante para verificar
print("Primeras filas del dataframe cruzado difuso Workspace:")
display(df_cruzado_difuso_workspace[:5])
print(f"\nNúmero de filas en el dataframe cruzado difuso: {len(df_cruzado_difuso_workspace)}")

Primeras filas del dataframe cruzado difuso Workspace:


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address,Mejor_coincidencia,Puntaje
0,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN,NaN,0.000000
1,67827639,LAMAS AROCA,LUNA ANGEL,LAMAS AROCA LUNA ANGEL,llamas@dummie.com,LAMAS AROC,95.238095
2,25808538,CUESTA ROCA,DEBORA BERNARDO,CUESTA ROCA DEBORA BERNARDO,NaN,NaN,0.000000
3,88753261,ANGEL IGLESIAS,CAROLINA ROSALINDA,ANGEL IGLESIAS CAROLINA ROSALINDA,NaN,NaN,0.000000
4,10709498,CARBO LASA,MARIA JOSE FELIPA,CARBO LASA MARIA JOSE FELIPA,NaN,NaN,0.000000



Número de filas en el dataframe cruzado difuso: 213


**Identificación de duplicados:** Utilizar una mascara booleana para indentificar los registros con valores en "APELLIDOS" duplicados. Como la búsqueda no se realizó con base al nombre completo, puede que existen personas con los mismos apellidos.

In [43]:
#Se busca duplicados en los "APELLIDOS" de la consulta 4.
duplicados_difuso_workspace = df_cruzado_difuso_workspace[df_cruzado_difuso_workspace.duplicated(subset=['DNI'], keep=False)]

#Imprimimos la cantidad de duplicados.
print(f"Registros duplicados en Consulta 4: {len(duplicados_difuso_workspace)}")

#Se muestran los valores duplicados, ordenandolos por nombre y apellido.
duplicados_difuso_workspace

Registros duplicados en Consulta 4: 0


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Email Address,Mejor_coincidencia,Puntaje


**Eliminación selectiva de duplicados (opcional depende de los resultados anteriores):** Tras identificar los registros duplicados con una cuenta de correo equivocado, se eliminan.

In [ ]:
#Lista de índices a eliminar
#indices_a_borrar = [1542,1543,1553] #

#Eliminamos las filas.
#df_cruzado_difuso_workspace = df_cruzado_difuso_workspace.drop(indices_a_borrar).reset_index(drop=True)

#Contamos la cantidad de filas en el df del cuarto cruce.
#len(df_cruzado_difuso_workspace)

1568

**Segmentación de resultados:** Crear un DataFrame de "No Encontrados" filtrando aquellos registros que resultaron con valor Nan en el campo de correo electrónico tras la Consulta 4. No hay valores vacíos, porque la tabla 2 procede de Workspace donde el campo correo electrónico siempre está lleno.

In [44]:
#Cantidad de registros con el campo "correo electrónico" vacío (no detectado por Nan).
vacio = (df_cruzado_difuso_workspace['Email Address'].str.strip()=="").sum()
print(f"Registros sin correo tras el cuarto cruce {vacio} no detectados por isnan.")

Registros sin correo tras el cuarto cruce 0 no detectados por isnan.


In [45]:
#Se filtran los registros con valores nulos de la columna Email Address.
nulos_difuso_workspace = df_cruzado_difuso_workspace[df_cruzado_difuso_workspace['Email Address'].isna()].copy()

#Imprimimos la cantidad de nulos en el cuarto cruce.
print(f"Registros sin correo tras el cuarto cruce {len(nulos_difuso_workspace)}")

Registros sin correo tras el cuarto cruce 139


###Notas

Luego de cada cruce, debe haber una **validación de integridad**, es decir realizar un conteo final para asegurar que no se hayan duplicado registros (mantener la cantidad original de la tabla 1). Asimismo, se debe hacer una **segmentación de resultados** que permita ver cuántos registros sin emparejar quedan.

**Consolidación final 1:** Unificar los correos obtenidos en las tres etapas (DNI, Nominal y Difuso) en una sola columna maestra dentro del DataFrame principal.

Este Dataframe se cruzará con el dataframe 1 mediante el campo DNI para obtener un archivo con la misma cantidad y orden de registros que el enviado por OGRH.

In [46]:
#Se copia el dataframe DNI-cruzado como base para la consolidación.
df_final_consolidado = df_cruzado_dni.copy()

#Se convierte el DNI en índice temporalmente para obtener los correos de los demás cruces.
df_final_consolidado = df_final_consolidado.set_index('DNI')

#Paso 1: Consolidar los correos electrónicos de coincidencia nominal (df_cruzado_nominal)
#La columna 'Email Address' en df_cruzado_nominal contiene los correos electrónicos encontrados mediante coincidencia nominal para el grupo 'nulos_dni'.
#Usaremos `fillna` en la columna 'Email Address' de df_final_consolidado, mapeando por DNI.
nominal_emails_map = df_cruzado_nominal.set_index('DNI')['Email Address']
df_final_consolidado['Email Address'] = df_final_consolidado['Email Address'].fillna(nominal_emails_map)

#Paso 2: Consolidar los correos electrónicos de coincidencia difusa (df_cruzado_difuso_da y df_cruzado_difuso_workspace)
#La columna 'Email Address' en los df_cruzado_difuso contiene los correos electrónicos encontrados mediante coincidencia difusa para el grupo 'nulos_nominal' y 'nulos_difuso_da'.
#Usaremos `fillna` en la columna 'Email Address' de df_final_consolidado, mapeando por DNI.
fuzzy_da_emails_map = df_cruzado_difuso_da.set_index('DNI')['Email Address']
df_final_consolidado['Email Address'] = df_final_consolidado['Email Address'].fillna(fuzzy_da_emails_map)

fuzzy_workspace_emails_map = df_cruzado_difuso_workspace.set_index('DNI')['Email Address']
df_final_consolidado['Email Address'] = df_final_consolidado['Email Address'].fillna(fuzzy_workspace_emails_map)

#Se vuelve el DNI como columna y restauramos el índice numérico.
df_final_consolidado = df_final_consolidado.reset_index()

#Renombramos a español la columna de correo electrónico.
df_final_consolidado.rename(columns={'Email Address': 'Correo electrónico'}, inplace=True)

#Se muestra el encabezado del DataFrame consolidado y cuentan los NaN restantes.
print("Consolidated DataFrame head:")
display(df_final_consolidado.head())
print(f"\nNúmero de filas en el DataFrame consolidado: {len(df_final_consolidado)}")
print(f"Número de NaN restantes en 'Correo electrónico': {df_final_consolidado['Correo electrónico'].isna().sum()}")

Consolidated DataFrame head:


,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Correo electrónico
0,85822413,CANTON AMAYA,MANUELA LEONARDO,CANTON AMAYA MANUELA LEONARDO,mcantón@dummie.com
1,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,dcalatayud@dummie.com
2,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN
3,13756670,RIVAS GIMENEZ,VIDAL DOLORES,RIVAS GIMENEZ VIDAL DOLORES,vrivas@dummie.com
4,56629389,CUERVO ROCAMORA,NOE LEOPOLDO,CUERVO ROCAMORA NOE LEOPOLDO,ncuervo@dummie.com



Número de filas en el DataFrame consolidado: 1000
Número de NaN restantes en 'Correo electrónico': 139


In [47]:
#Se hace un merge para traer cruzar la tabla consolidada y la tabla enviada por OGRH.
df_consolidado_completo = pd.merge(
    df_1,
    df_final_consolidado[['DNI','Correo electrónico']],
    left_on='DNI',
    right_on='DNI',
    how='left'
)

#Imprimimos la tabla, la cantidad de registros y la cantidad de nulos.
display(df_consolidado_completo.head())
print(f"\n N° registros de la tabla consolidada: {len(df_consolidado_completo)}")
print(f"\n N° registros nulos de la tabla consolidada: {df_consolidado_completo['Correo electrónico'].isna().sum()}")

,DNI,APELLIDOS,NOMBRES,APELLIDOS Y NOMBRES,Correo electrónico
0,85822413,CANTON AMAYA,MANUELA LEONARDO,CANTON AMAYA MANUELA LEONARDO,mcantón@dummie.com
1,99529224,CALATAYUD BONET,DOMINGA TELMO,CALATAYUD BONET DOMINGA TELMO,dcalatayud@dummie.com
2,29958839,GUERRERO VAZQUEZ,ELIGIA AMANDO,GUERRERO VAZQUEZ ELIGIA AMANDO,NaN
3,13756670,RIVAS GIMENEZ,VIDAL DOLORES,RIVAS GIMENEZ VIDAL DOLORES,vrivas@dummie.com
4,56629389,CUERVO ROCAMORA,NOE LEOPOLDO,CUERVO ROCAMORA NOE LEOPOLDO,ncuervo@dummie.com



 N° registros de la tabla consolidada: 1050

 N° registros nulos de la tabla consolidada: 145


##Exportación

**Generación de archivos:** Exportar el resultado final a un archivo .xlsx o .csv.

In [48]:
df_consolidado_completo.to_excel("Correos_Personal_Vigente.xlsx", index=False)

**Reporte de excepciones:** Generar un archivo secundario con los registros que no pudieron ser emparejados ni siquiera con Fuzzy Matching para una revisión manual rápida.

In [49]:
#Se exporta los registros que no pudieron ser emparejados a un archivo Excel.
nulos_difuso_workspace.to_excel("Registros_No_Emparejados.xlsx", index=False)